In [ ]:
import requests
import pandas as pd
import json
import os
import numpy as np
from time import sleep
from scipy.stats import norm
import warnings

# Configurações de exibição e avisos
pd.options.display.max_rows = 100
pd.options.display.max_columns = 100
warnings.filterwarnings('ignore')


In [ ]:
# ---------------------------------------------------
# 1. PARÂMETROS E PREÇOS TETO
# ---------------------------------------------------
SELIC = 0.1475  # Taxa livre de risco padronizada para 2026

precos_teto_suno_dividendos = {
    "WIZC3": 10.00, "BBSE3": 35.50, "BBAS3": 25.00, "UNIP6": 70.00, "SEER3": 14.00, "VALE3": 75.00,
    "PETR4": 34.00, "AXIA6": 43.80, "TUPY3": 21.00, "AGRO3": 27.50, "EGIE3": 28.60, "ITSA4": 9.50,
}

carteira_PM = {
    "ABEV3": 10.00, "B3SA3": 11.08, "BBAS3": 21.96, "BBSE3": 32.00,
    "EGIE3": 28, "FLRY3": 15.60, "HYPE3": 29.31, "ITSA4": 9.41,
    "KLBN11": 18.58, "LEVE3": 33.92, "PETR4": 30.06, "TAEE11": 38.28,
    "UNIP6": 50.99, "VALE3": 58.13, "RADL3": 25
}

precos_teto = {k: precos_teto_suno_dividendos.get(k, v) for k, v in carteira_PM.items()}

In [ ]:
# ---------------------------------------------------
# 2. DOWNLOAD E ATUALIZAÇÃO (CORREÇÃO JSON)
# ---------------------------------------------------
if not os.path.exists("dbJson"):
    os.makedirs("dbJson")

def atualizarDados(ativo):
    url = f"https://storage.googleapis.com/api-cdn-eaglesystem/api/{ativo.upper()}"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return f"{ativo}: erro HTTP {response.status_code}"
        
        # Validação de conteúdo JSON para evitar erro 'Expecting value'
        content = response.text.strip()
        if not (content.startswith('{') or content.startswith('[')):
            return f"{ativo}: Erro - Conteúdo não é JSON válido"

        dados = response.json()
        with open(f"dbJson/{ativo}.json", "w", encoding="utf-8") as arq:
            json.dump(dados, arq, indent=2)
        return f"{ativo}: atualizado"
    except Exception as e:
        return f"{ativo}: erro -> {e}"

for ativo in precos_teto.keys():
    print(atualizarDados(ativo))


ABEV3: atualizado
B3SA3: atualizado
BBAS3: atualizado
BBSE3: atualizado
EGIE3: atualizado
FLRY3: atualizado
HYPE3: atualizado
ITSA4: atualizado
KLBN11: atualizado
LEVE3: atualizado
PETR4: atualizado
TAEE11: atualizado
UNIP6: atualizado
VALE3: atualizado
RADL3: atualizado


In [ ]:
# ---------------------------------------------------
# 3. PROCESSAMENTO DOS DADOS (DATAFRAME)
# ---------------------------------------------------
def dataFrameUnico(ativo):
    with open(f"dbJson/{ativo}.json", "r", encoding="utf-8") as arq:
        dados = json.load(arq)

    preco_atual = dados["asset"]["close"]
    linhas = []

    for serie in dados["series"]:
        vencimento = serie.get("due_date")
        dias = serie.get("days_to_maturity")

        for strike in serie["strikes"]:
            strike_price = strike["strike"]
            for tipo in ["call", "put"]:
                opt = strike[tipo]
                if opt:
                    linhas.append({
                        "ativo": ativo, "tipo": tipo.upper(), "vencimento": vencimento,
                        "dias": dias, "strike": strike_price, "symbol": opt["symbol"],
                        "bid": opt["bid"], "ask": opt["ask"], "volume": opt["volume"],
                        "delta": opt["bs"]["delta"], "theta": opt["bs"]["theta"],
                        "vol": opt["bs"]["volatility"], "poe": opt["bs"]["poe"],
                        "preco_atual": preco_atual
                    })
    return pd.DataFrame(linhas)

todos = []
for ativo, pteto in precos_teto.items():
    try:
        df = dataFrameUnico(ativo)
        df["preco_teto"] = pteto
        todos.append(df)
    except: pass

df_final = pd.concat(todos, ignore_index=True)

In [ ]:
# ---------------------------------------------------
# 4. CÁLCULOS E FILTROS DE PUTS
# ---------------------------------------------------
puts = df_final[df_final["tipo"] == "PUT"].copy()
puts = puts[puts['vol'] > 0].copy() # Evita erro matemático

puts["retorno"] = (puts["bid"] / (puts["strike"] - puts['bid'])) * 100
puts["retorno_mes"] = puts["retorno"] * (30 / puts["dias"].replace(0, 1))
puts["dist_strike"] = ((puts["strike"] / puts["preco_atual"]) - 1) * 100

# Black-Scholes para PUTS
T = (puts['dias'] / 252).replace(0, 0.001)
sigma = puts['vol'] / 100
d1 = (np.log(puts['preco_atual'] / puts['strike']) + (SELIC + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)
puts['black_scholes'] = (puts['strike'] * np.exp(-SELIC * T) * norm.cdf(-d2) - puts['preco_atual'] * norm.cdf(-d1)).round(2)
puts['desvio_bs'] = (puts['bid'] - puts['black_scholes']).round(2)

aporte = 5000
puts['cotas'] = np.floor((aporte / puts['strike'])/100) * 100
puts['premio X cotas'] = puts['cotas'] * puts['bid']

# Ranking e Score
filtro_put = puts[

    (puts['dias'].between(15, np.inf)) & 
    (puts['dist_strike'] <= 0) & 
    (puts['retorno_mes'] >= 0.95) 


].copy()

if not filtro_put.empty:
    filtro_put['score'] = (
        filtro_put['dist_strike'].rank(ascending=True) * 4 + 
        filtro_put['retorno_mes'].rank(ascending=False) * 5
    )
    print("\n--- MELHORES OPÇÕES DE VENDA DE PUT ---")
    
    display(filtro_put[

        [
        'ativo','symbol', 'tipo', 'strike', 'preco_atual',  
        'preco_teto', 'dist_strike', 'bid', 'ask', 'black_scholes',
        'volume', 'delta', 'theta', 'vol', 'poe',  'retorno', 'retorno_mes',
        'desvio_bs', 'dias', 'vencimento','score', 'cotas', 'premio X cotas'
        ]
        
        ].sort_values('score').head(30))

    


--- MELHORES OPÇÕES DE VENDA DE PUT ---


,ativo,symbol,tipo,strike,preco_atual,preco_teto,dist_strike,bid,ask,black_scholes,volume,delta,theta,vol,poe,retorno,retorno_mes,desvio_bs,dias,vencimento,score,cotas,premio X cotas
21749,RADL3,RADLT162,PUT,16.28,17.11,25.00,-4.850964,0.33,0.00,0.53,13600,-0.281440,-0.008292,41.018,32.98,2.068966,1.773399,-0.20,35,2026-08-21,56.0,300.0,99.0
2001,B3SA3,B3SAT139,PUT,13.76,14.78,11.08,-6.901218,0.23,0.41,0.28,42900,-0.218992,-0.006024,36.235,25.92,1.699926,1.457080,-0.05,35,2026-08-21,71.0,300.0,69.0
2005,B3SA3,B3SAT144,PUT,14.26,14.78,11.08,-3.518268,0.33,0.71,0.42,31300,-0.308372,-0.006706,35.496,35.54,2.368988,2.030561,-0.09,35,2026-08-21,79.0,300.0,99.0
21751,RADL3,RADLT165,PUT,16.53,17.11,25.00,-3.389831,0.40,0.00,0.62,27500,-0.319779,-0.008530,41.001,37.07,2.479851,2.125587,-0.22,35,2026-08-21,82.0,300.0,120.0
12993,PETR4,PETRT379,PUT,36.86,38.28,34.00,-3.709509,0.71,0.88,0.74,39500,-0.255490,-0.010742,28.911,28.73,1.964039,1.683462,-0.03,35,2026-08-21,88.0,100.0,71.0
13989,PETR4,PETRV383,PUT,35.80,38.28,34.00,-6.478579,1.16,1.90,1.35,6600,-0.193866,-0.005239,37.616,23.42,3.348730,1.376190,-0.19,73,2026-10-16,90.0,100.0,116.0
2003,B3SA3,B3SAT141,PUT,14.01,14.78,11.08,-5.209743,0.22,0.65,0.35,59500,-0.262124,-0.006439,36.155,30.60,1.595359,1.367451,-0.13,35,2026-08-21,103.0,300.0,66.0
7843,EGIE3,EGIET297,PUT,29.75,32.15,28.60,-7.465008,0.41,0.65,0.45,25100,-0.171555,-0.009402,32.918,20.10,1.397410,1.197780,-0.04,35,2026-08-21,123.0,100.0,41.0
18207,VALE3,VALET774,PUT,77.46,79.03,75.00,-1.986587,1.40,2.79,1.80,104400,-0.312312,-0.021236,27.087,34.55,1.840652,1.577702,-0.40,35,2026-08-21,144.0,0.0,0.0
13001,PETR4,PETRT389,PUT,37.86,38.28,34.00,-1.097179,1.06,1.74,1.07,150300,-0.352536,-0.010972,28.522,38.88,2.880435,2.468944,-0.01,35,2026-08-21,145.0,100.0,106.0


In [ ]:
# ---------------------------------------------------
# 5. CÁLCULOS E FILTROS DE CALLS (COVERED CALL)
# ---------------------------------------------------
calls = df_final[df_final["tipo"] == "CALL"].copy()
calls = calls[(calls['vol'] > 0) & (calls['dias'] > 0)].copy()

T_c = calls["dias"] / 252
sigma_c = calls["vol"] / 100
d1_c = (np.log(calls["preco_atual"] / calls["strike"]) + (SELIC + sigma_c**2 / 2) * T_c) / (sigma_c * np.sqrt(T_c))
d2_c = d1_c - sigma_c * np.sqrt(T_c)

calls["black_scholes"] = (calls["preco_atual"] * norm.cdf(d1_c) - calls["strike"] * np.exp(-SELIC * T_c) * norm.cdf(d2_c)).round(2)
calls["retorno_anual"] = (calls["bid"] / calls["preco_atual"]) * (365 / calls["dias"]) * 100
calls["dist_strike"] = ((calls["strike"] / calls["preco_atual"]) - 1) * 100

filtro_call = calls[
    (calls['dist_strike'] >= 0) & (calls['retorno_anual'] >= 6) & (calls['volume'] > 0)
].copy()

if not filtro_call.empty:
    print("\n--- MELHORES OPÇÕES DE CALL COBERTA ---")
    display(filtro_call.sort_values('retorno_anual', ascending=False).head(20))


--- MELHORES OPÇÕES DE CALL COBERTA ---


,ativo,tipo,vencimento,dias,strike,symbol,bid,ask,volume,delta,theta,vol,poe,preco_atual,preco_teto,black_scholes,retorno_anual,dist_strike
2926,BBAS3,CALL,2026-07-10,5,20.23,BBASG203W2,0.15,0.30,136900,0.414188,-0.032909,22.700,40.02,20.00,25.00,0.18,54.750000,1.150000
11968,PETR4,CALL,2026-07-17,10,38.36,PETRG395,0.51,1.11,480600,0.539907,-0.050700,22.213,51.95,38.28,34.00,0.75,48.628527,0.208986
11970,PETR4,CALL,2026-07-17,10,38.61,PETRG397,0.45,0.77,451800,0.489472,-0.049813,20.020,46.90,38.28,34.00,0.56,42.907524,0.862069
6082,BBSE3,CALL,2026-07-17,10,38.68,BBSEG386,0.45,1.00,106600,0.563437,-0.043334,18.085,54.74,38.67,35.50,0.67,42.474787,0.025860
1662,B3SA3,CALL,2026-07-17,10,14.96,B3SAG152,0.17,0.46,494000,0.478218,-0.024376,34.316,45.07,14.78,11.08,0.36,41.982409,1.217862
11972,PETR4,CALL,2026-07-17,10,38.86,PETRG39,0.43,0.54,505600,0.439528,-0.048311,20.950,41.94,38.28,34.00,0.48,41.000522,1.515152
13006,PETR4,CALL,2026-08-21,35,38.61,PETRH397,1.50,2.03,27900,0.569127,-0.032506,24.892,53.11,38.28,34.00,1.65,40.864308,0.862069
254,ABEV3,CALL,2026-07-17,10,16.38,ABEVG164,0.18,0.50,42600,0.513362,-0.021398,21.380,49.29,16.29,10.00,0.28,40.331492,0.552486
13004,PETR4,CALL,2026-08-21,35,38.36,PETRH394,1.43,1.93,121000,0.595539,-0.032737,25.180,55.80,38.28,34.00,1.80,38.957307,0.208986
1664,B3SA3,CALL,2026-07-17,10,15.21,B3SAG153,0.15,0.37,355300,0.384521,-0.022784,34.329,35.83,14.78,11.08,0.26,37.043302,2.909337
